<a href="https://colab.research.google.com/github/k-nikhil-coder/100-day-ai-engineer/blob/main/Day09_Regression_ScikitLearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
California Housing Price Prediction
===================================
A comparison between standard Multiple Linear Regression and Polynomial Regression
(Degree 2) using select features from the California Housing dataset.
"""

from typing import Tuple

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


def load_and_prepare_data(
    features: list[str], test_size: float = 0.2, random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Load the California housing dataset and split into train/test sets."""
    dataset = fetch_california_housing(as_frame=True)
    df = dataset.frame

    X = df[features]
    y = df["MedHouseVal"]

    return train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )


def evaluate_model(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    """Calculate and display MAE, RMSE, and R² evaluation metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"--- {name} ---")
    print(f"Mean Absolute Error (MAE)  : {mae:.4f}")
    print(f"Root Mean Squared Error    : {rmse:.4f}")
    print(f"R² Score                   : {r2:.4f}\n")


def main() -> None:
    # -------------------------------------------------------------------------
    # 1. Load Data
    # -------------------------------------------------------------------------
    selected_features = ["MedInc", "HouseAge", "AveRooms"]
    X_train, X_test, y_train, y_test = load_and_prepare_data(selected_features)

    # -------------------------------------------------------------------------
    # 2. Feature Scaling
    # -------------------------------------------------------------------------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # -------------------------------------------------------------------------
    # 3. Model 1: Linear Regression
    # -------------------------------------------------------------------------
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    y_pred_lr = lr_model.predict(X_test_scaled)

    # -------------------------------------------------------------------------
    # 4. Model 2: Polynomial Regression (Degree 2)
    # -------------------------------------------------------------------------
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)

    poly_model = LinearRegression()
    poly_model.fit(X_train_poly, y_train)
    y_pred_poly = poly_model.predict(X_test_poly)

    # -------------------------------------------------------------------------
    # 5. Model Evaluation
    # -------------------------------------------------------------------------
    print("=" * 45)
    print("           MODEL EVALUATION METRICS          ")
    print("=" * 45 + "\n")

    evaluate_model("Linear Regression Baseline", y_test, y_pred_lr)
    evaluate_model("Polynomial Regression (Degree 2)", y_test, y_pred_poly)

    # -------------------------------------------------------------------------
    # 6. Cross-Validation Check
    # -------------------------------------------------------------------------
    cv_scores = cross_val_score(
        lr_model, X_train_scaled, y_train, cv=5, scoring="r2"
    )

    print("--- Cross-Validation ---")
    print(
        f"Linear Regression 5-Fold R² Mean: {cv_scores.mean():.4f} "
        f"(± {cv_scores.std():.4f})\n"
    )


if __name__ == "__main__":
    main()

           MODEL EVALUATION METRICS          

--- Linear Regression Baseline ---
Mean Absolute Error (MAE)  : 0.6033
Root Mean Squared Error    : 0.8117
R² Score                   : 0.4972

--- Polynomial Regression (Degree 2) ---
Mean Absolute Error (MAE)  : 0.5895
Root Mean Squared Error    : 0.8013
R² Score                   : 0.5101

--- Cross-Validation ---
Linear Regression 5-Fold R² Mean: 0.5134 (± 0.0120)

